In [1]:
import sys
import os
from pydataxm.pydatasimem import ReadSIMEM, CatalogSIMEM
import pandas as pd

def get_df(id, fecha_inicio, fecha_final, nombre_csv="archivo.csv"):
    # --- Obtención de datos ---
    catalogo = CatalogSIMEM(catalog_type='Datasets')
    df_catalogo = catalogo.get_data()

    dataset_id = id
    fecha_fin = fecha_final
    simem = ReadSIMEM(dataset_id, fecha_inicio, fecha_fin)
    df_general = simem.main()

    # --- Calcular ruta DOS niveles arriba, sin crear carpetas ---
    # Si __file__ no existe (Jupyter/REPL), usamos el cwd
    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()  # entorno interactivo

    carpeta_dos_arriba = os.path.abspath(os.path.join(base_dir, ".."))

    # Verificación: no crear carpetas; si no existe, fallar con mensaje claro
    if not os.path.isdir(carpeta_dos_arriba):
        raise FileNotFoundError(
            f"La carpeta dos niveles arriba no existe o no es accesible: {carpeta_dos_arriba}"
        )

    ruta_csv = os.path.join(carpeta_dos_arriba, nombre_csv)

    # Manejo defensivo si la librería devuelve None
    if df_general is None:
        df_general = pd.DataFrame()
        print("Advertencia: ReadSIMEM.main() devolvió None. Se guarda DataFrame vacío.")

    # Guardar CSV (no se crean carpetas)
    df_general.to_csv(ruta_csv, index=False)
    print(f"CSV guardado en: {ruta_csv}")

    return df_general

In [2]:
import requests
from typing import List, Any, Dict, Union

def obtener_namecolumns(dataset_id: str, url_template: str, timeout: int = 200) -> List[str]:
    
    # 1) Construir la URL reemplazando el placeholder exactamente como lo tienes en tu código
    if "{dataset_id}" not in url_template:
        raise ValueError("El url_template debe contener el placeholder {dataset_id}")
    url = url_template.format(dataset_id=dataset_id)

    # 2) Llamar al endpoint
    resp = requests.get(url, timeout=timeout)
    resp.raise_for_status()

    # 3) Parsear JSON
    try:
        data = resp.json()
    except ValueError as e:
        raise ValueError(f"La respuesta no es JSON válido. Error: {e}")

    if data is None:
        raise ValueError("La respuesta JSON está vacía (None).")

    # 4) Buscar recursivamente 'Columns' y extraer 'nameColumn'
    def _find_namecolumns(obj: Union[Dict[str, Any], List[Any]]) -> List[str]:
        found: List[str] = []
        columns_keys_lower = {"columns"}  # case-insensitive ('Columns' o 'columns')

        def _walk(node: Any):
            if isinstance(node, dict):
                for k, v in node.items():
                    # ¿Esta clave es 'Columns' (sin importar mayúsculas)?
                    if str(k).lower() in columns_keys_lower:
                        # v puede ser list o dict. Extraer 'nameColumn' en formatos comunes.
                        if isinstance(v, list):
                            for item in v:
                                if isinstance(item, dict) and "nameColumn" in item:
                                    found.append(item["nameColumn"])
                        elif isinstance(v, dict):
                            # Buscar en subclaves típicas
                            candidates_list = None
                            for ck in ("items", "data", "list", "values"):
                                if ck in v and isinstance(v[ck], list):
                                    candidates_list = v[ck]
                                    break
                            if candidates_list:
                                for item in candidates_list:
                                    if isinstance(item, dict) and "nameColumn" in item:
                                        found.append(item["nameColumn"])
                            else:
                                # Último recurso: barrer el dict y listas internas
                                for subv in v.values():
                                    if isinstance(subv, dict) and "nameColumn" in subv:
                                        found.append(subv["nameColumn"])
                                    elif isinstance(subv, list):
                                        for item in subv:
                                            if isinstance(item, dict) and "nameColumn" in item:
                                                found.append(item["nameColumn"])
                    # Seguir recorriendo el árbol
                    _walk(v)
            elif isinstance(node, list):
                for it in node:
                    _walk(it)

        _walk(obj)

        # Devolver únicos conservando el orden
        seen = set()
        unique = []
        for x in found:
            if x not in seen:
                unique.append(x)
                seen.add(x)
        return unique

    return _find_namecolumns(data)


##EJEMPLO DE USO###
#dataset_id = "75f675"
#url_template = "https://www.simem.co/backend-files/api/detalle-datos-publicos?datasetId={dataset_id}"


#columnas = obtener_namecolumns(dataset_id, url_template)
#print("nameColumn encontrados:", columnas)

In [4]:

id = "1eee63"
inicio = "2025-01-01"
fin = "2025-05-01"
url = "https://www.simem.co/backend-files/api/detalle-datos-publicos?datasetId={dataset_id}"

print("Etapa1")
get_df(id,inicio,fin)
print("Etapa2")
colums = obtener_namecolumns(id, url)
print(colums)



Etapa1
****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Aportes Energético Mediano Plazo"
****************************************************************************************************
Inicio consulta sincronica
Creacion url: 0.0
Extraccion de registros: 6.859184741973877
End of data extracting process
****************************************************************************************************
CSV guardado en: d:\ambiente\escritorio\COSITAS_DE_PY\ProyectoPracticas\archivo.csv
Etapa2
['FechaFin', 'SemanaEstudio', 'Etapa', 'FechaPublicacion', 'NombreCaso', 'CodigoAreaOperativa', 'AportesProyectados', 'AnioEstudio', 'Caso', 'FechaInicio']


In [ ]:
df_data = pd.read_csv("../archivo.csv", index_col= False)
df_data.head(5)
df_data.describe

,FechaPublicacion,Etapa,FechaInicio,FechaFin,Caso,NombreCaso,CodigoAreaOperativa,AportesProyectados,AnioEstudio,SemanaEstudio
0,2024-12-29,22,2025-05-26,2025-06-01,Estocastico,Percentil 5,Are0010,170885712.0,2024,52
1,2024-12-29,22,2025-05-26,2025-06-01,Estocastico,Promedio,Are0010,290804288.0,2024,52
2,2024-12-29,22,2025-05-26,2025-06-01,Caso4,HEsperado,Are0128,46515400.0,2024,52
3,2024-12-29,22,2025-05-26,2025-06-01,Caso2,H14-16,Are0128,26573598.0,2024,52
4,2024-12-29,22,2025-05-26,2025-06-01,Caso1,H93-95,Are0128,48851884.0,2024,52
